# Python for Research Data — Part 1
### Foundations: Colab · variables · numpy & pandas · loops · **writing your own functions**

---

You do not need to become a programmer. You need to stop doing the same thing 40 times by hand.

That is what this session is about: **Part 1 builds the machinery** — the five or six Python ideas that cover 90% of a research analysis. **Part 2** uses that machinery to build publication-quality figures.

| Part | Section | Time |
|---|---|---|
| **0** | Why Python, and why Colab | 5 min |
| **1** | Colab tour: runtime, packages, your files | 8 min |
| **2** | Basic operations: print, variables, maths, lists | 10 min |
| **3** | `numpy` and `pandas` — the two libraries you will actually live in | 12 min |
| **4** | `for` loops — do it once, repeat it 200 times | 8 min |
| **5** | **Functions — how to design one** | 17 min |
| 6 | Put it together + bring your own data | take-home |
| 7 | Exercises (with solutions) | take-home |

---

### The dataset

**COIL 1999 European river water-quality survey** — 200 water samples from rivers across Europe, each analysed for chemistry and algal content.

| Column | Meaning |
|---|---|
| `season`, `river_size`, `fluid_velocity` | Sampling context |
| `max_pH`, `min_O2` | pH and dissolved oxygen |
| `chloride`, `nitrate`, `ammonium`, `orthophosphate`, `phosphate` | Nutrient chemistry (mg L⁻¹) |
| **`chlorophyll_a`** | **Chlorophyll-a concentration (µg L⁻¹) — our headline measurement** |
| `alga_1` … `alga_7` | Abundance of seven algal groups |

> **Our question all session:** *does slow-moving water breed more algae?*

*Source: [UCI Machine Learning Repository — COIL 1999](https://archive.ics.uci.edu/ml/machine-learning-databases/coil-mld/) · water-quality survey.*

---
# Part 0 · Why Python, why Colab  ⏱ 5 min

## What Python buys you

You already analyse data. The question is not *Excel vs Python* — it is **how many times do you want to do this by hand?**

| The task | In a spreadsheet | In Python |
|---|---|---|
| Recompute after new samples arrive | Re-do every step, hope you remember them | Re-run one cell |
| The same plot for 12 treatment groups | 12 charts, 12 chances to mis-click | A `for` loop, 4 lines |
| "How exactly did you get this number?" | Scroll through cells and try to reconstruct | The code *is* the answer |
| A reviewer asks for log axes on all panels | Edit every chart | Change one word in one function |
| 300 CSV files from an instrument | 300 opens | A loop over a folder |

Three words for why this matters in science:

- **Automation** — write the recipe once; the computer repeats it without getting tired or bored.
- **Efficiency** — one changed number re-runs the whole analysis in seconds.
- **Reproducibility** — the script is a complete, honest record of what you did. Future-you, your PI and your reviewers can all replay it.

## Where we are going

By the end of Part 1 you will be able to read — and write — this:

```python
rivers  = load_rivers()                                    # your function
summary = summarise_by(rivers, "fluid_velocity", "chlorophyll_a")   # your function
plot_group_box(rivers, "fluid_velocity", "chlorophyll_a")           # your function
```

Three lines. Each one is a function **you** will write today. That is the whole point of Part 1: not memorising commands, but packaging your analysis into pieces you can re-use on next month's data.

---
# Part 1 · Colab in 8 minutes  ⏱ 8 min

**Google Colab** is a Jupyter notebook that runs on Google's computers, in your browser. Nothing to install, and everyone in the room has the identical setup.

## The interface

| Thing | What it does |
|---|---|
| **Cell** | A box holding either code or text. Everything happens in cells. |
| **▶ / `Shift + Enter`** | Run the cell and move to the next one |
| **`Ctrl/Cmd + Enter`** | Run the cell and stay put |
| **`+ Code` / `+ Text`** | Add a new cell |
| **Runtime → Restart session** | Wipe everything and start clean — your first move when things get weird |
| **Files 📁 (left sidebar)** | Temporary storage. **Deleted when you close the tab** — download anything you want to keep |

## Three rules that explain 90% of beginner confusion

1. **Order matters.** Cells run in the order *you press play*, not top to bottom. If you skip a cell, later cells will not find what they need.
2. **Memory persists.** A variable made in one cell is available in every other cell — until you restart the runtime.
3. **The last line prints itself.** Anything else needs `print()`.

## 1.1 The runtime — what you are actually using

When you press ▶ for the first time, Google hands you a **runtime**: a temporary Linux computer in a data centre, with Python already installed.

| | |
|---|---|
| **What you get** | ~12 GB RAM, ~100 GB disk, a CPU (free GPU/TPU under *Runtime → Change runtime type*) |
| **How long** | Disconnects after ~90 min idle, and after ~12 h in any case |
| **What survives a disconnect** | **Nothing.** Variables, uploaded files, installed packages — all gone. Your notebook text and code survive because they live in Google Drive |
| **Restart session** | Clears all variables, keeps your files |
| **Disconnect and delete runtime** | Throws the whole machine away — the true fresh start |

> **The practical rule:** anything you care about — a cleaned table, a figure — must be **saved and downloaded**, or written straight back to Google Drive. Treat the runtime as a bench you have to clear at the end of the day.

The `!` prefix runs a **shell** command instead of Python, so you can inspect that machine directly:

In [ ]:
# These are Linux commands — they work on the Colab runtime,
# and will say "command not found" if you run this notebook on your own laptop.
!python --version
!nvidia-smi -L      # lists a GPU, if you attached one
!free -g            # RAM, in gigabytes

## 1.2 Installing dependencies

A **library** (or package) is code someone else wrote so you do not have to. The five we need are **already installed in Colab**:

| Library | Nickname | Job |
|---|---|---|
| `pandas` | `pd` | Spreadsheets in code — load, filter, group |
| `numpy` | `np` | Fast maths on arrays |
| `matplotlib` | `plt` | The drawing engine |
| `seaborn` | `sns` | Statistical plots, beautifully, in one line |
| `scipy.stats` | — | Hypothesis tests |

Anything else you install yourself with `pip`, Python's package manager:

```python
!pip install scikit-posthocs        # a package Colab does not ship with
!pip install "pandas==2.2.0"        # a specific version, when a tutorial needs one
!pip install package --quiet        # hide the wall of output
```

> **Two things to know.** Installs vanish when the runtime restarts, so keep the `!pip install` line at the top of your notebook. And if pip says *"restart the runtime"*, do it — Python only notices a new version of an already-loaded library after a restart.

Running the install for a package you already have is harmless — pip just says *already satisfied*.

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy --quiet

# `import` loads a library into this session. `as pd` gives it a short nickname.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu

print("✅ pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)

In [ ]:
# House style for every figure in this notebook — set it once, never think about it again.
plt.rcParams.update({
    "font.size"        : 11,
    "axes.linewidth"   : 0.9,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "figure.dpi"       : 110,
    "savefig.bbox"     : "tight",
    "savefig.dpi"      : 300,      # journal submission resolution
})

# Chlorophyll-inspired palettes, re-used in every figure below
FLOW_ORDER = ["Low", "Medium", "High"]
FLOW_PAL   = {"Low": "#08605f", "Medium": "#3fa34d", "High": "#c5d86d"}     # deep water → algal green
SIZE_ORDER = ["Small", "Medium", "Large"]
SIZE_PAL   = {"Small": "#ffd166", "Medium": "#ef8354", "Large": "#8d3b72"}  # amber → plum

print("✅ Style set. Every figure in this notebook will match.")

## 1.3 Getting your data in

Four ways, in rough order of how often you will use them:

| Method | Code | When |
|---|---|---|
| **From a URL** | `pd.read_csv("https://…")` | Public data. No download, always the current version |
| **Upload a file** | `files.upload()` | A one-off spreadsheet from your instrument |
| **Drag into 📁** | *(no code)* | Same, but drag-and-drop into the Files sidebar, then `pd.read_csv("myfile.csv")` |
| **Mount Google Drive** | `drive.mount("/content/drive")` | Data you return to every week |

```python
# Google Drive — everything in your Drive appears at /content/drive/MyDrive/
from google.colab import drive
drive.mount("/content/drive")
df = pd.read_csv("/content/drive/MyDrive/lab_data/plate_reader.csv")
```

> **Getting a raw CSV URL from GitHub:** open the file in the repo → click **Raw** → copy the address bar. That is the link `pd.read_csv()` wants.

The upload cell below is **Colab-only** (it opens a file picker in the browser). Skip it for now — we come back to it in Part 6 with your own data.

In [ ]:
# 📌 COLAB ONLY — run this when you want to analyse your own spreadsheet.
# from google.colab import files
# uploaded = files.upload()                 # opens a file picker
# filename = list(uploaded.keys())[0]
# mydata   = pd.read_csv(filename)          # or pd.read_excel(filename)
# print("Loaded", filename, "→", mydata.shape)

---
# Part 2 · Basic operations  ⏱ 10 min

Everything in Python is built from four ideas: **values**, **variables**, **collections of values**, and **things you do to them**.

## 2.1 Variables and types

In [ ]:
# Anything after a # is a comment — Python ignores it. Comments are for humans.

experiment = "Chlorophyll survey"      # text  (str)
n_samples  = 200                       # whole number (int)
mean_chl   = 13.5                      # decimal (float)
is_pilot   = False                     # True/False (bool)

print(experiment, "|", n_samples, "samples")
print("Mean chlorophyll-a:", mean_chl, "µg/L")

# f-strings drop variables straight into text — the {} are filled in
print(f"{experiment}: {n_samples} samples, mean {mean_chl:.1f} µg/L")

# type() tells you what kind of thing you are holding — useful when maths misbehaves
print(type(n_samples), type(mean_chl), type(experiment))

> **`=` is not equals.** It means *"put the thing on the right into the name on the left"*. Testing equality is `==`. That single/double confusion is one of the most common beginner bugs.
>
> **Naming:** lowercase, underscores, descriptive — `mean_chl`, not `x`. Three months from now `x` will mean nothing to you.

## 2.2 Maths

Python is a calculator that remembers.

In [ ]:
blank      = 0.08          # absorbance of the blank
absorbance = 0.62

corrected = absorbance - blank
print("Corrected A:", corrected)

# The operators
print("add        ", 7 + 2)
print("subtract   ", 7 - 2)
print("multiply   ", 7 * 2)
print("divide     ", 7 / 2)      # always a decimal
print("power      ", 7 ** 2)
print("remainder  ", 7 % 2)      # useful for 'every Nth item'

# Beer–Lambert:  c = A / (ε · l)
epsilon, path = 0.0899, 1.0                   # L µg⁻¹ cm⁻¹, cm
conc = corrected / (epsilon * path)
print(f"\nChlorophyll-a = {conc:.2f} µg/mL")

# Rounding and formatting: :.2f = 2 decimals, :.1e = scientific, :, = thousands separator
print(f"{conc:.2f}  |  {conc:.1e}  |  {123456.789:,.1f}")

## 2.3 Lists — several values in one name

In [ ]:
# A list holds several values in order. Counting starts at ZERO.
readings = [2.1, 5.6, 20.7, 110.5, 0.4]

print("First reading :", readings[0])      # 0 = first!
print("Third reading :", readings[2])
print("Last reading  :", readings[-1])     # -1 = last
print("First three   :", readings[0:3])    # a slice: from 0, up to (not including) 3
print("How many      :", len(readings))
print("Biggest       :", max(readings))
print("Average       :", sum(readings) / len(readings))

readings.append(8.8)                       # add to the end
print("After append  :", readings)

# A loop repeats an action for every item — more on this in Part 4
for r in readings:
    if r > 10:
        print(f"  ⚠️  {r} µg/L — bloom territory")

## 2.4 Make the computer do something visible 🐍

Nothing here is new — a list, an index, a `for` loop, an `if` — but put together they *move*.

`clear_output(wait=True)` wipes the previous frame before printing the next one, which is all an animation is: draw, wait, wipe, draw again.

In [ ]:
import time
from IPython.display import clear_output

WIDTH = 34
pos, direction = 0, 1

for frame in range(48):
    track = ["·"] * WIDTH          # 34 empty positions
    track[pos] = "●"               # put the dot at the current position

    clear_output(wait=True)
    print(" " + "".join(track))    # "".join turns the list into one string
    print(f" frame {frame:2d} · position {pos:2d}")
    time.sleep(0.04)

    pos = pos + direction                      # move
    if pos == 0 or pos == WIDTH - 1:           # hit a wall? turn around
        direction = -direction

print("\n🎉 A list you edited 48 times, printed 48 times. That is an animation.")

---
# Part 3 · numpy and pandas  ⏱ 12 min

| | Holds | Use it for |
|---|---|---|
| **numpy** `np.array` | Numbers, all the same type, in a grid | Fast maths on a whole column at once |
| **pandas** `pd.DataFrame` | A table: named columns, mixed types, an index | Everything you would otherwise do in a spreadsheet |

A pandas DataFrame is built *on top of* numpy arrays — each column is essentially an array wearing a name tag.

## 3.1 numpy: maths without loops

With a list you would loop. With an array you just write the maths, and numpy applies it to every element.

In [ ]:
abs_raw = np.array([0.62, 0.48, 0.91, 0.15, 0.73])     # five wells on a plate
blank   = 0.08

corrected = abs_raw - blank          # subtracts from EVERY element — no loop
conc      = corrected / 0.0899       # divides every element

print("raw       :", abs_raw)
print("corrected :", corrected.round(3))
print("conc µg/mL:", conc.round(2))

print("\nmean", conc.mean().round(2),
      "| sd", conc.std(ddof=1).round(2),
      "| min", conc.min().round(2),
      "| max", conc.max().round(2))

# Handy array builders
print("\nnp.arange(0, 10, 2) :", np.arange(0, 10, 2))       # 0 to 10, step 2
print("np.linspace(0, 1, 5):", np.linspace(0, 1, 5))        # 5 evenly spaced values
print("np.log10([1, 10, 100]):", np.log10([1, 10, 100]))    # log transform, elementwise

### Why it matters: speed

"Vectorised" means *the loop happens inside numpy's C code instead of yours*. On one million numbers:

In [ ]:
n = 1_000_000

t0 = time.perf_counter()
squares_loop = [x * x for x in range(n)]        # Python doing it one at a time
t1 = time.perf_counter()

arr = np.arange(n)
t2 = time.perf_counter()
squares_np = arr ** 2                            # numpy doing it all at once
t3 = time.perf_counter()

print(f"Python loop : {(t1 - t0) * 1000:7.1f} ms")
print(f"numpy       : {(t3 - t2) * 1000:7.1f} ms")
print(f"numpy is roughly {(t1 - t0) / max(t3 - t2, 1e-9):.0f}× faster")

> **The rule of thumb:** if you find yourself looping over rows of a dataset to do arithmetic, there is almost always a vectorised one-liner. Loop over **groups, columns, files and figures** — not over individual numbers.

## 3.2 pandas: your data as a table

Build a small DataFrame by hand first, so you can see what one *is*: a dictionary of columns.

In [ ]:
plate = pd.DataFrame({
    "well"      : ["A1", "A2", "A3", "B1", "B2", "B3"],
    "treatment" : ["Control", "Control", "Control", "Nitrate", "Nitrate", "Nitrate"],
    "absorbance": [0.62, 0.48, 0.55, 0.91, 0.88, 1.02],
})

plate["chl_ugml"] = (plate["absorbance"] - 0.08) / 0.0899   # a whole new column, one line

display(plate)

print("\nOne column (a Series):");  print(plate["chl_ugml"].round(2).to_list())
print("Mean by treatment:");        print(plate.groupby("treatment")["chl_ugml"].mean().round(2))

### The five verbs you will use forever

| Spreadsheet habit | pandas |
|---|---|
| Look at the top of the sheet | `df.head()` |
| Select a column | `df["col"]` |
| Filter rows | `df[df["col"] > 10]` |
| Sort | `df.sort_values("col", ascending=False)` |
| **PivotTable** | **`df.groupby("category")["value"].mean()`** |

> `groupby` is the one to internalise: **split** the table by a category, **apply** a calculation to each piece, **combine** the answers. It is a PivotTable you can re-run.

## 3.3 Load the real dataset

`pd.read_csv()` reads a CSV straight from a web address — no download, no file browser. But this file has three traps, and real data usually does:

| Trap | Symptom | Fix |
|---|---|---|
| No header row | Your first sample becomes the column names | `header=None, names=COLS` |
| Missing values written as `XXXXXXX` | A numeric column arrives as text; `.mean()` raises `TypeError` | `na_values="XXXXXXX"` |
| 17 rows have a comma missing in the source file | Values fuse (`2.822008777.59961`) and every later column shifts left | Those — and only those — rows lose `alga_7`: drop where `alga_7` is missing |

The third one is the real lesson: **the error message was a symptom, not the disease.** Forcing the column to numeric would have "fixed" the error and left 17 rows of silently wrong data in your analysis. *(Part 2 of this series takes the full tour of reading error messages.)*

In [ ]:
URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/coil-mld/analysis.data"
COLS = [
    "season", "river_size", "fluid_velocity",
    "max_pH", "min_O2", "chloride", "nitrate", "ammonium",
    "orthophosphate", "phosphate", "chlorophyll_a",
    "alga_1", "alga_2", "alga_3", "alga_4", "alga_5", "alga_6", "alga_7",
]

raw = pd.read_csv(URL, header=None, names=COLS, na_values="XXXXXXX")

print("Shape (rows, columns):", raw.shape)
raw.head()

In [ ]:
# Four commands to run on EVERY new dataset, before anything else.
print("1 · SIZE:", raw.shape, "\n")

print("2 · TYPES — anything 'object' that should be a number is a landmine:")
print(raw.dtypes.to_string(), "\n")

print("3 · MISSING VALUES per column:")
print(raw.isna().sum()[raw.isna().sum() > 0].to_string(), "\n")

print("4 · NUMERIC SANITY CHECK — look for impossible minima and maxima:")
display(raw.describe().round(2))

### Clean it

Four steps: drop the corrupted rows, convert `nitrate` to numbers for real, strip the invisible padding from the text columns (`"small_"` is **not** equal to `"small"` — that one silently returns zero rows), and drop samples with no chlorophyll reading.

In [ ]:
damaged = raw["alga_7"].isna()                 # exactly and only the shifted rows
print("Damaged rows:", damaged.sum(), "of", len(raw))

rivers = raw[~damaged].copy()                  # ~ means NOT. .copy() = work on our own copy
rivers["nitrate"] = pd.to_numeric(rivers["nitrate"])

# Category values are padded to six characters: "small_", "high__". Invisible, fatal to ==.
for col in ["season", "river_size", "fluid_velocity"]:
    rivers[col] = rivers[col].str.replace("_", "", regex=False).str.capitalize()

rivers = rivers.dropna(subset=["chlorophyll_a"]).reset_index(drop=True)

print("Clean dataset:", rivers.shape[0], "samples")
print("river_size    :", list(rivers["river_size"].unique()))
print("fluid_velocity:", list(rivers["fluid_velocity"].unique()))
rivers.head()

In [ ]:
# groupby = PivotTable. Split by flow velocity, apply four calculations, combine.
summary = (
    rivers.groupby("fluid_velocity")["chlorophyll_a"]
          .agg(n="count", median="median", mean="mean", max="max")
          .reindex(FLOW_ORDER)          # our order, not alphabetical
          .round(2)
)
display(summary)

ratio = summary.loc["Low", "median"] / summary.loc["High", "median"]
print(f"\nMedian chlorophyll-a is {ratio:.0f}x higher in slow water than in fast water.")

---
# Part 4 · `for` loops  ⏱ 8 min

> **Write it once. Let the computer repeat it.**

```python
for item in collection:      # ← the colon is compulsory
    do_something(item)       # ← the indentation is compulsory (4 spaces)
```

Python uses **indentation** instead of brackets to decide what is inside the loop. Lines that line up under the `for` run every time; the first line that unindents runs once, at the end.

In [ ]:
# 1 · Loop over a list of things
for group in FLOW_ORDER:
    subset = rivers[rivers["fluid_velocity"] == group]
    print(f"{group:7s} n = {len(subset):3d}   median = {subset['chlorophyll_a'].median():6.2f} µg/L")

print()

# 2 · enumerate() when you need a counter as well as the item
for i, group in enumerate(FLOW_ORDER):
    print(f"  position {i} → {group}")

print()

# 3 · Loop over columns: the same check applied to every nutrient
for col in ["nitrate", "ammonium", "orthophosphate", "chlorophyll_a"]:
    values = rivers[col]
    print(f"{col:16s} missing {values.isna().sum():2d}  |  median {values.median():8.2f}")

In [ ]:
# 4 · Build a list as you go — collect results instead of printing them
results = []                                   # start empty

for group in FLOW_ORDER:
    values = rivers.loc[rivers["fluid_velocity"] == group, "chlorophyll_a"]
    results.append({                           # append one dict per group
        "flow"  : group,
        "n"     : len(values),
        "median": round(values.median(), 2),
        "mean"  : round(values.mean(), 2),
    })

display(pd.DataFrame(results))                 # a list of dicts becomes a DataFrame

In [ ]:
# 5 · Loop over figures — three panels, one block of code, guaranteed identical styling
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)

for ax, group in zip(axes, FLOW_ORDER):        # zip pairs them up: (axes[0], "Low"), …
    values = rivers.loc[rivers["fluid_velocity"] == group, "chlorophyll_a"]
    ax.hist(values, bins=np.logspace(-0.7, 2.1, 14), color=FLOW_PAL[group], edgecolor="white")
    ax.set_xscale("log")
    ax.set_title(f"{group} flow  (n = {len(values)})", fontsize=10, loc="left")
    ax.set_xlabel("Chlorophyll-a (µg L$^{-1}$)")

axes[0].set_ylabel("Samples")
fig.suptitle("Same code, three groups — this is what a loop is for", x=0.02, ha="left", y=1.04)
plt.show()

> **When NOT to loop.** Looping over *rows* to do arithmetic is slow and hard to read. Compare:
>
> ```python
> logs = []                                  # ❌ don't
> for value in rivers["chlorophyll_a"]:
>     logs.append(np.log10(value))
>
> rivers["log_chl"] = np.log10(rivers["chlorophyll_a"])   # ✅ do
> ```
>
> Loop over **groups, columns, files, figures**. Vectorise over **numbers**.

In [ ]:
rivers["log_chl"] = np.log10(rivers["chlorophyll_a"])    # one line, 171 rows
rivers[["chlorophyll_a", "log_chl"]].head(3).round(3)

## 4.1 🐍 Everything so far, in fifteen lines

A list (the snake), indexing, a `for` loop, an `if`, and `%` for wrapping around. Watch it eat.

In [ ]:
WIDTH = 34
snake     = [3, 2, 1, 0]      # positions of each segment, head first
direction = 1
food      = 20

for frame in range(90):
    track = ["·"] * WIDTH
    track[food] = "✱"
    for segment in snake[1:]:
        track[segment] = "○"
    track[snake[0]] = "●"

    clear_output(wait=True)
    print(" " + "".join(track))
    print(f" frame {frame:2d} · length {len(snake)}")
    time.sleep(0.05)

    head = snake[0] + direction
    if head >= WIDTH or head < 0:              # bounce off the wall
        direction = -direction
        head = snake[0] + direction
    snake.insert(0, head)                      # new head at the front

    if head == food:
        food = (food + 13) % WIDTH             # % wraps the new food back inside the track
    else:
        snake.pop()                            # no food eaten → drop the tail, length unchanged

print(f"\n🐍 Final length: {len(snake)} — and not one line of it was written twice.")

---
# Part 5 · Functions — how to design one  ⏱ 17 min

A loop repeats a block of code **here**. A function lets you repeat it **anywhere, on anything, later** — including on the data you have not collected yet.

## 5.1 Anatomy

```python
def sem(values):                     # def · name · (inputs) · colon
    """Standard error of the mean."""    # docstring: what it does, for a human
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]        # the body — indented
    return values.std(ddof=1) / np.sqrt(len(values))   # what comes back out
```

| Part | Why it matters |
|---|---|
| `def name(...)` | The name is how you will call it. Use a **verb**: `load_`, `clean_`, `plot_`, `summarise_` |
| **Parameters** | Everything the function needs, handed in explicitly |
| **Docstring** | One line saying what it does. Future-you will read this, not the code |
| `return` | The answer travels back to the caller. A function that only `print`s gives you nothing to use next |

## 5.2 The design checklist

Before you write the body, answer four questions:

| Question | For a scientific analysis |
|---|---|
| **1. What goes in?** | Usually a `pd.DataFrame`, a column name, or a `np.array` / `pd.Series` of measurements |
| **2. What comes out?** | A number, a cleaned DataFrame, a summary table, or a plotted `Axes` |
| **3. What is the one job?** | If your description needs the word "and", it is probably two functions |
| **4. What should be adjustable?** | Anything you might change next time — group order, colours, log axis — becomes a parameter **with a sensible default** |

> **The rule that saves you most pain:** pass the data in, hand the result back. A function that reaches out and grabs a global `df` works exactly once — on today's dataset, in today's notebook, in today's cell order.

## 5.3 Type 1 — a number in, a number out

The simplest useful function: a calculation you keep re-typing. `np.asarray()` at the top means it accepts a **list, a numpy array or a pandas Series** without complaining — design for the inputs you will actually meet.

In [ ]:
def sem(values):
    """
    Standard error of the mean, ignoring missing values.

    Parameters
    ----------
    values : list, np.array or pd.Series of numbers

    Returns
    -------
    float
    """
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]                       # drop NaN, or the answer is NaN
    return v.std(ddof=1) / np.sqrt(len(v))    # ddof=1 = sample SD


def cv_percent(values):
    """Coefficient of variation (SD / mean) as a percentage."""
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    return 100 * v.std(ddof=1) / v.mean()


# Test on something you can check by hand BEFORE trusting it on real data
print("sem([2, 4, 4, 4, 5, 5, 7, 9]) =", round(sem([2, 4, 4, 4, 5, 5, 7, 9]), 4))

# Now the real thing — same function, three different input types
low = rivers.loc[rivers["fluid_velocity"] == "Low", "chlorophyll_a"]
print("\nSlow-flow rivers:")
print(f"  mean   {low.mean():.2f} ± {sem(low):.2f} µg/L (SEM, n = {len(low)})")
print(f"  CV     {cv_percent(low):.0f}%  ← hugely variable, as blooms tend to be")

In [ ]:
def sig_stars(p):
    """Convert a p-value to the usual star notation."""
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"


def draw_bracket(ax, x1, x2, y, text, lw=1.2, color="black", fontsize=10):
    """Draw a significance line from x1 to x2 at height y, labelled with `text`."""
    ax.plot([x1, x2], [y, y], lw=lw, color=color, clip_on=False)
    ax.text((x1 + x2) / 2, y, text, ha="center", va="bottom",
            fontsize=fontsize, color=color)


fast = rivers.loc[rivers["fluid_velocity"] == "High", "chlorophyll_a"]
U, p_flow = mannwhitneyu(low, fast)
print(f"Low vs High flow: U = {U:.0f}, p = {p_flow:.2e} → {sig_stars(p_flow)}")

> **Why `sig_stars` is a function and not three lines you retype:** the day a reviewer asks you to use a different threshold, you change it in **one** place and every figure in the notebook updates.

## 5.4 Type 2 — a DataFrame in, a DataFrame out

The summary table we built in Part 3 was hard-coded to one column of one dataset. Turn it into a function and it works on any table, any grouping, any measurement.

Notice the parameters: `data`, `group`, `value` are required; `order` and `decimals` have **defaults**, so the simple call stays simple.

In [ ]:
def summarise_by(data, group, value, order=None, decimals=2):
    """
    Summary statistics of `value`, split by `group`.

    Parameters
    ----------
    data     : pd.DataFrame
    group    : str   name of the categorical column
    value    : str   name of the numeric column
    order    : list of category names, or None for whatever pandas finds
    decimals : int   rounding for the output table

    Returns
    -------
    pd.DataFrame — one row per group: n, median, mean, sem
    """
    out = (
        data.dropna(subset=[value])
            .groupby(group)[value]
            .agg(n="count", median="median", mean="mean", sem=sem)
    )
    if order is not None:
        out = out.reindex(order)
    return out.round(decimals)


display(summarise_by(rivers, "fluid_velocity", "chlorophyll_a", order=FLOW_ORDER))
display(summarise_by(rivers, "river_size",     "chlorophyll_a", order=SIZE_ORDER))
display(summarise_by(rivers, "season",         "nitrate"))      # different column entirely

Three different questions, one function, zero copy-paste. **That is the return on the ten minutes you spent writing it.**

## 5.5 Type 3 — a DataFrame in, a figure out

This is the workhorse of a scientific analysis, and it has one non-obvious design decision: **the `ax=None` parameter**.

| `ax` | What happens |
|---|---|
| Not given (`None`) | The function makes its own figure — perfect while you are exploring |
| An `Axes` handed in | The function draws **into your figure**, wherever you put it — perfect for multi-panel publication figures |

That single parameter is the difference between a plot you can only look at and a panel you can assemble. Note also that the function **returns the `ax`**, so the caller can keep customising afterwards.

In [ ]:
def plot_group_box(data, group, value, order=None, palette=None, ax=None,
                   log=True, ylabel=None, show_stats=True):
    """
    Box + raw-points plot of `value` split by `group`, with a Kruskal-Wallis test.

    Parameters
    ----------
    data       : pd.DataFrame
    group      : str   categorical column → x axis
    value      : str   numeric column     → y axis
    order      : list  category order (default: order of appearance)
    palette    : dict  {category: colour} (default: seaborn's)
    ax         : matplotlib Axes, or None to create a new figure
    log        : bool  log y axis — essential when the data span orders of magnitude
    ylabel     : str   axis label (default: the column name)
    show_stats : bool  add the Kruskal-Wallis result to the title

    Returns
    -------
    matplotlib Axes — so you can keep customising after the call
    """
    d = data.dropna(subset=[group, value])
    if order is None:
        order = list(d[group].unique())
    if ax is None:                                   # ← no axes given: make our own
        fig, ax = plt.subplots(figsize=(5.4, 4.2))

    sns.boxplot(data=d, x=group, y=value, order=order, hue=group,
                palette=palette, legend=False, dodge=False, width=0.55,
                fliersize=0, linewidth=1.2, ax=ax)   # dodge=False keeps boxes centred
    sns.stripplot(data=d, x=group, y=value, order=order,
                  color="#22223b", size=3.5, alpha=0.55, jitter=0.18, ax=ax)

    means = d.groupby(group)[value].mean().reindex(order)
    ax.scatter(range(len(order)), means, marker="D", s=55, color="white",
               edgecolor="#22223b", zorder=10, label="mean")

    if log:
        ax.set_yscale("log")
    ax.set_xlabel(group.replace("_", " ").capitalize())
    ax.set_ylabel(value.replace("_", " ") if ylabel is None else ylabel)
    ax.legend(frameon=False, loc="upper right", fontsize=9)

    if show_stats:
        H, p = kruskal(*[d.loc[d[group] == g, value] for g in order])
        ax.set_title(f"Kruskal–Wallis  H = {H:.1f},  p = {p:.1e}  {sig_stars(p)}",
                     loc="left", fontsize=10)
    return ax


# Call 1 — no ax: the function builds its own figure
plot_group_box(rivers, "fluid_velocity", "chlorophyll_a",
               order=FLOW_ORDER, palette=FLOW_PAL,
               ylabel="Chlorophyll-a (µg L$^{-1}$)")
plt.show()

In [ ]:
# Call 2 — same function, different columns, different palette. One line.
plot_group_box(rivers, "river_size", "chlorophyll_a",
               order=SIZE_ORDER, palette=SIZE_PAL,
               ylabel="Chlorophyll-a (µg L$^{-1}$)")
plt.show()

### The payoff: a composite figure in six lines

Because the plotting logic lives in a function that accepts an `ax`, a two-panel journal figure costs almost nothing. This is exactly how the publication figures in **Part 2** are built.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw={"wspace": 0.28})

plot_group_box(rivers, "fluid_velocity", "chlorophyll_a", order=FLOW_ORDER,
               palette=FLOW_PAL, ax=axes[0], ylabel="Chlorophyll-a (µg L$^{-1}$)")
plot_group_box(rivers, "river_size", "chlorophyll_a", order=SIZE_ORDER,
               palette=SIZE_PAL, ax=axes[1], ylabel="")

# The function handed the axes back, so we can keep editing them.
# Note get_title(loc="left") — a title set on the left must be read back from the left.
for ax, label in zip(axes, ["(A)  By flow velocity", "(B)  By river size"]):
    ax.set_title(f"{label}\n{ax.get_title(loc='left')}", loc="left", fontsize=10)

axes[0].set_ylim(top=rivers["chlorophyll_a"].max() * 6)      # headroom for the bracket
axes[0].legend(frameon=False, loc="lower left", fontsize=9)  # out of the bracket's way
axes[1].set_ylim(axes[0].get_ylim())                         # same measurement → same axis

# A significance line between the two extreme groups, using our two helpers.
# Compute the test HERE rather than relying on a p-value left over in another cell.
_, p_low_high = mannwhitneyu(rivers.loc[rivers["fluid_velocity"] == "Low",  "chlorophyll_a"],
                             rivers.loc[rivers["fluid_velocity"] == "High", "chlorophyll_a"])
draw_bracket(axes[0], 0, 2, y=rivers["chlorophyll_a"].max() * 1.6,
             text=sig_stars(p_low_high), fontsize=13)

fig.suptitle("Slow, large rivers breed algae", x=0.09, y=1.06, ha="left", fontsize=13)
fig.savefig("fig_part1_composite.png", dpi=300, bbox_inches="tight")   # Files 📁 → download
plt.show()

> **Reading it:** median chlorophyll-a is ~10× higher in slow-flowing rivers than fast ones (20.7 vs 2.1 µg L⁻¹, p ≈ 2 × 10⁻⁹), and rises with river size (2.1 → 6.4 → 14.7 µg L⁻¹, p ≈ 1 × 10⁻⁶). Algae need time in one place to divide; fast water flushes them downstream before a bloom can establish.
>
> **The white diamonds sit above the medians** in every group — the signature of a right-skewed distribution, where a few extreme blooms drag the mean up.

## 5.6 Type 4 — a function that hides the mess

Everything we did in Part 3 to clean this dataset was worth doing **once**. Wrap it up, and loading clean data becomes a single line — forever.

In [ ]:
def load_rivers(url=URL, drop_missing_chl=True):
    """
    Load and clean the COIL 1999 river survey.

    Handles the three quirks of this file: no header row, missing values coded
    'XXXXXXX', and 17 rows corrupted by a missing comma (detectable because they
    are the only rows with no alga_7 value).

    Parameters
    ----------
    url              : str   where the raw data lives
    drop_missing_chl : bool  drop samples with no chlorophyll reading

    Returns
    -------
    pd.DataFrame — clean, all-numeric where it should be, tidy category labels
    """
    df = pd.read_csv(url, header=None, names=COLS, na_values="XXXXXXX")

    df = df[~df["alga_7"].isna()].copy()               # drop the shifted rows
    df["nitrate"] = pd.to_numeric(df["nitrate"])

    for col in ["season", "river_size", "fluid_velocity"]:
        df[col] = df[col].str.replace("_", "", regex=False).str.capitalize()

    if drop_missing_chl:
        df = df.dropna(subset=["chlorophyll_a"])

    return df.reset_index(drop=True)


rivers = load_rivers()
print("Loaded and cleaned:", rivers.shape[0], "samples ·", rivers.shape[1], "columns")
rivers.head(3)

## 5.7 Four mistakes worth avoiding

| ❌ | ✅ |
|---|---|
| `def plot():` then using the global `df` inside | `def plot(data):` — hand the data in |
| The function `print`s the answer | The function `return`s it, and the caller prints if it wants to |
| One function that loads **and** cleans **and** plots | Three functions you can test and re-use separately |
| Editing the DataFrame that was passed in | `df = data.copy()` first — never surprise your caller by mutating their data |

Two more habits:

- **Test on something you can check by hand** before trusting a function on 200 samples.
- **Name honestly.** `plot_group_box` says what you get. `analysis2_final` does not.

> A function does three things at once: it stops you repeating yourself, it gives a name to a step of your analysis, and it makes that step **testable**. Everything else in Python is optional. This is not.

---
# Part 6 · Put it together, on your data

Your toolbox from today — one line each:

```python
rivers  = load_rivers()                                                   # load + clean
summary = summarise_by(rivers, "fluid_velocity", "chlorophyll_a", FLOW_ORDER)   # the numbers
ax      = plot_group_box(rivers, "fluid_velocity", "chlorophyll_a")       # the figure
```

Now swap in your own spreadsheet. **Run the upload cell in Colab**, then the checklist, then the plot.

In [ ]:
# 📌 COLAB ONLY — uncomment and run. Click "Choose Files", pick a .csv or .xlsx.
# from google.colab import files
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# mydata   = pd.read_csv(filename) if filename.endswith(".csv") else pd.read_excel(filename)
# print("Loaded", filename, "→", mydata.shape)
# mydata.head()

In [ ]:
def first_five_minutes(data, name="dataset"):
    """Everything you should check before trusting a new table. Prints; returns nothing."""
    print(f"── {name} ─────────────────────────────")
    print("1 · SIZE      :", data.shape)
    print("2 · COLUMNS   :", list(data.columns))
    print("\n3 · TYPES — 'object' where you expect a number is a landmine:")
    print(data.dtypes.to_string())
    print("\n4 · MISSING VALUES:")
    miss = data.isna().sum()
    print(miss[miss > 0].to_string() if (miss > 0).any() else "   none 🎉")
    print("\n5 · NUMERIC SANITY CHECK — impossible minima or maxima?")
    display(data.describe().round(2))
    print("6 · TEXT COLUMNS — look for near-duplicates like 'Control' vs 'control ':")
    for c in data.select_dtypes(exclude="number").columns[:6]:
        print(f"   {c:18s} {data[c].nunique():3d} unique → {list(data[c].dropna().unique()[:6])}")


first_five_minutes(rivers, "river survey")     # 🎯 swap `rivers` for `mydata`

In [ ]:
# 🎯 YOUR TURN — two column names, and the function you wrote does the rest.
GROUP   = "river_size"          # ← your categorical column
MEASURE = "chlorophyll_a"       # ← your numeric column
DATA    = rivers                # ← your DataFrame (e.g. mydata)

display(summarise_by(DATA, GROUP, MEASURE))

ax = plot_group_box(DATA, GROUP, MEASURE, log=True)
ax.set_title(f"{MEASURE} by {GROUP}\n{ax.get_title(loc='left')}", loc="left", fontsize=10)
plt.show()

---
# Part 7 · Exercises 🎯

A **new dataset**: 62 leaves in three glasshouses, measured under different light intensities and CO₂ levels.

| Column | Meaning |
|---|---|
| `glasshouse` | Which of three glasshouses (A/B/C) |
| `CO2level` | CO₂ treatment: low / medium / high |
| **`light`** | **Light intensity (µmol photons m⁻² s⁻¹)** |
| `tempDiff` | Leaf temperature **minus** air temperature (°C) |
| `airTemp`, `vapPress` | Air temperature (°C), vapour pressure |

**The biology:** a leaf in bright light absorbs energy faster than it can use or shed it, so it runs hotter than the surrounding air. Transpiration is the leaf's air-conditioning.

> **Your question: does higher light intensity make leaves run hotter than the air around them?**

The real test of today's session is that **the functions you wrote for river water work on leaf temperature without modification.**

*Source: [DAAG R package](https://vincentarelbundock.github.io/Rdatasets/doc/DAAG/leaftemp.html) — glasshouse leaf-temperature experiment.*

In [ ]:
LEAF_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/DAAG/leaftemp.all.csv"
leaf = pd.read_csv(LEAF_URL)

first_five_minutes(leaf, "leaf temperature")

### 🎯 Task 1 — a calculation function

Write `mean_pm_sem(values)` that returns a **formatted string** like `"1.24 ± 0.09"`, using the `sem()` function you already have. Then use it on `tempDiff`.

*Hint: `f"{m:.2f} ± {s:.2f}"`.*

In [ ]:
# 🎯 YOUR TURN
def mean_pm_sem(values):
    """Return 'mean ± sem' as a formatted string."""
    # your code here
    pass


# print("tempDiff:", mean_pm_sem(leaf["tempDiff"]), "°C")

### 🎯 Task 2 — reuse `summarise_by`

`light` is a continuous measurement, so there is nothing to group by yet. Bin it into three equal-sized bands with `pd.qcut(column, 3, labels=[...])`, then call `summarise_by()` on the result.

Fill in the blanks marked `____`.

In [ ]:
# 🎯 YOUR TURN
leaf["light_band"] = pd.qcut(leaf["____"], 3, labels=["Low", "Medium", "High"])

display(summarise_by(leaf, "____", "____", order=["Low", "Medium", "High"]))

### 🎯 Task 3 — reuse `plot_group_box`

Plot `tempDiff` by `light_band`. Two things to think about:

1. `tempDiff` contains **negative and near-zero values**, so a log axis will fail — which parameter turns it off?
2. Add a dashed horizontal line at 0 (leaf temperature = air temperature) with `ax.axhline(0, color="grey", ls="--", lw=1)`. The function returns the `ax`, so you can do this after the call.

A light-themed palette: `{"Low": "#2c3e70", "Medium": "#e8a33d", "High": "#f6e27a"}` (dusk → noon).

In [ ]:
# 🎯 YOUR TURN

### 🎯 Task 4 — write a function of your own

Write `plot_light_response(data, ax=None)` that produces the figure from Task 3 in one call — the same design pattern: data in, `ax=None`, return the `ax`. Then build a **two-panel composite**: leaf temperature on the left, river chlorophyll on the right.

**Stretch:** add a `kind="box"` parameter that switches to `sns.violinplot` when you pass `kind="violin"`.

In [ ]:
# 🎯 YOUR TURN

---
---
# 🔑 Solutions

*Try them first — the struggle is where the learning happens.*

In [ ]:
# ── Task 1 ──────────────────────────────────────────────────────────────
def mean_pm_sem(values):
    """Return 'mean ± sem' as a formatted string."""
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    return f"{v.mean():.2f} ± {sem(v):.2f}"


print("tempDiff:", mean_pm_sem(leaf["tempDiff"]), "°C")
print("chlorophyll-a:", mean_pm_sem(rivers["chlorophyll_a"]), "µg/L")

In [ ]:
# ── Task 2 ──────────────────────────────────────────────────────────────
# pd.qcut splits a continuous column into bands with equal NUMBERS of samples.
leaf["light_band"] = pd.qcut(leaf["light"], 3, labels=["Low", "Medium", "High"])

BANDS = ["Low", "Medium", "High"]
display(summarise_by(leaf, "light_band", "tempDiff", order=BANDS))
display(summarise_by(leaf, "light_band", "light",    order=BANDS))   # check the bands are sensible

In [ ]:
# ── Task 3 ──────────────────────────────────────────────────────────────
# log=False, because tempDiff goes negative — log10 of a negative number is undefined.
LIGHT_PAL = {"Low": "#2c3e70", "Medium": "#e8a33d", "High": "#f6e27a"}   # dusk → noon

ax = plot_group_box(leaf, "light_band", "tempDiff", order=BANDS,
                    palette=LIGHT_PAL, log=False,
                    ylabel="Leaf − air temperature (°C)")
ax.axhline(0, color="grey", ls="--", lw=1)          # leaf temperature == air temperature
ax.set_title(f"Do bright leaves run hot?\n{ax.get_title(loc='left')}", loc="left", fontsize=10)
plt.show()

In [ ]:
# ── Task 4 ──────────────────────────────────────────────────────────────
def plot_light_response(data, ax=None, kind="box"):
    """
    Leaf−air temperature across light-intensity bands.

    data : pd.DataFrame with 'light_band' and 'tempDiff'
    ax   : Axes, or None for a new figure
    kind : "box" or "violin"

    Returns the Axes.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(5.4, 4.2))

    if kind == "violin":
        sns.violinplot(data=data, x="light_band", y="tempDiff", order=BANDS,
                       hue="light_band", palette=LIGHT_PAL, legend=False,
                       dodge=False, inner="quart", cut=0, ax=ax)
    else:
        plot_group_box(data, "light_band", "tempDiff", order=BANDS,
                       palette=LIGHT_PAL, log=False, ax=ax,
                       ylabel="Leaf − air temperature (°C)")

    ax.axhline(0, color="grey", ls="--", lw=1)
    ax.set_xlabel("Light intensity band")
    ax.set_ylabel("Leaf − air temperature (°C)")
    return ax


fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw={"wspace": 0.3})

plot_light_response(leaf, ax=axes[0])
plot_group_box(rivers, "fluid_velocity", "chlorophyll_a", order=FLOW_ORDER,
               palette=FLOW_PAL, ax=axes[1], ylabel="Chlorophyll-a (µg L$^{-1}$)")

for ax, label in zip(axes, ["(A)  Leaves under light", "(B)  Rivers by flow"]):
    ax.set_title(f"{label}\n{ax.get_title(loc='left')}", loc="left", fontsize=10)

fig.suptitle("Two experiments, one set of functions", x=0.09, y=1.06, ha="left", fontsize=13)
plt.show()

---
# What you learned today

| Idea | The one line to remember |
|---|---|
| **Colab runtime** | A temporary computer. Restart clears variables; download anything you want to keep |
| **`!pip install`** | Installs a package for this session only — keep it at the top of the notebook |
| **Variables** | `name = value`. `=` assigns, `==` compares |
| **f-strings** | `f"mean {m:.2f} µg/L"` — formatting without string glue |
| **Lists** | Ordered, zero-indexed, `[-1]` is the last |
| **numpy** | Maths on a whole array at once — no loop, 10–100× faster |
| **pandas** | `df.head()` · `df["col"]` · `df[df["col"] > 10]` · **`df.groupby(...)`** = PivotTable |
| **`for` loops** | Loop over groups, columns, files and figures. Vectorise over numbers |
| **Functions** | Data in → result out. `ax=None` makes a plot function composable |
| **Cleaning** | `header=None, names=` · `na_values=` · check dtypes before you trust a mean |

### The function design checklist, one more time

1. **What goes in?** — DataFrame, column names, arrays. Pass them; never reach for a global.
2. **What comes out?** — `return` a number, a table or an `Axes`. Don't just print.
3. **One job.** — If the description needs "and", split it.
4. **Defaults for everything adjustable.** — `order=None`, `palette=None`, `ax=None`, `log=True`.
5. **Docstring.** — One line. Future-you is the reader.
6. **Test it on something you can check by hand.**

### Coming in Part 2

Publication-quality figures with exactly these tools: violin + box with mean ± SEM, grouped bars with individual data points, significance brackets, and multi-panel composite figures at `dpi=300` — modelled on a real *Journal of Bacteriology* paper.

---

## 🧯 When you get stuck — the five-step routine

1. **Read the bottom line** of the traceback. Error type + message. Nothing above it matters yet.
2. **Find the `---->` arrow.** That is the failing line.
3. **Print the ingredients.** `print(df.columns)`, `df.dtypes`, `df["col"].unique()`, `len(df)`. Most bugs die here.
4. **Shrink the problem.** Break one long line into three short ones and run them separately.
5. **Search the message.** Paste the error type + message into Google. Someone has had it before. Trim anything specific to your own files first.

| Error | Plain English | Usual cause |
|---|---|---|
| `SyntaxError` | "I can't even read this." | Missing `)`, `:` or quote |
| `NameError` | "I've never heard of that." | Typo, or you skipped the cell that made it |
| `KeyError` | "That column doesn't exist." | Wrong column name — check spelling **and case** |
| `TypeError` | "Wrong kind of thing." | Doing maths on text |
| `ValueError` | "Right kind, impossible value." | Converting `"XXXXXXX"` to a number |
| `AttributeError` | "That thing has no such command." | Typo in a method name (`.avg()` instead of `.mean()`) |
| **No error at all** | **The code ran and gave you the wrong answer.** | **Only you can catch this — sanity-check every number** |

---

## Resources

| Resource | What it covers |
|---|---|
| [pandas — 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) | The official quick tour |
| [seaborn gallery](https://seaborn.pydata.org/examples/index.html) | Every plot type, with the code under it |
| [Scientific Visualization: Python + Matplotlib](https://github.com/rougier/scientific-visualization-book) | Free book on publication-quality figures |
| [Nature Methods — Points of Significance](https://www.nature.com/collections/qghhqm/pointsofsignificance) | Statistics for biologists, free article series |
| [Rdatasets](https://vincentarelbundock.github.io/Rdatasets/articles/data.html) | Hundreds of small real datasets to practise on |

### Saving figures

```python
fig.savefig("figure.png",  dpi=300, bbox_inches="tight")   # raster, most journals
fig.savefig("figure.pdf",  dpi=300, bbox_inches="tight")   # vector, editable in Illustrator
fig.savefig("figure.tiff", dpi=600, bbox_inches="tight")   # high-res raster for submission
```

Then open the **Files 📁** sidebar in Colab and download it — the runtime will not keep it for you.

---
*Data: UCI COIL 1999 river water-quality survey · DAAG leaf temperature experiment*